# 🏁 Notebook 4: Concurrent Retries & Race Conditions

What if a client retries *before* the first request has finished? Both requests arrive at the server at essentially the same time, each carrying the same idempotency key. Naively, they'll both read the idempotency table, see nothing, and each charge the customer.

Real systems solve this with two tools:

1. **An atomic `INSERT`** with a `UNIQUE` constraint — only one inserter wins.
2. **An `IN_PROGRESS` marker** so a concurrent retry can wait/reject instead of running the side effect again.

We'll demonstrate the problem first, then the fix.

## 🛠️ Setup

```bash
cd 04-patterns/idempotency
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 🟥 BAD: check-then-act race

Two threads interleave between *read* and *write* — both think they're first.

In [1]:
import threading, time, uuid

class RacyService:
    def __init__(self):
        self.balances = {'alice': 100}
        self.idem = {}

    def charge(self, key, amount):
        if key in self.idem:
            return ('REPLAY', self.idem[key])
        # simulate some work — this is where the window opens for a race
        time.sleep(0.05)
        self.balances['alice'] -= amount
        result = {'balance': self.balances['alice']}
        self.idem[key] = result
        return ('FRESH', result)

svc = RacyService()
k = str(uuid.uuid4())
results = []

def worker():
    results.append(svc.charge(k, 10))

threads = [threading.Thread(target=worker) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

print('results    :', results)
print('final bal  :', svc.balances['alice'], '← charged multiple times for ONE logical request 😱')


results    : [('FRESH', {'balance': 90}), ('FRESH', {'balance': 80}), ('FRESH', {'balance': 70}), ('FRESH', {'balance': 60}), ('FRESH', {'balance': 50})]
final bal  : 50 ← charged multiple times for ONE logical request 😱


Five concurrent retries, five charges. The `if key in self.idem` check is too late — all threads passed it before any thread had finished writing.

## 🟩 GOOD: atomic INSERT wins the race

Instead of *check-then-act*, we attempt an atomic `INSERT` of an `IN_PROGRESS` marker keyed by the idempotency key. At most one insert succeeds — the winner runs the side effect; losers poll for the winner's result.

In [2]:
import sqlite3, json

conn = sqlite3.connect(':memory:', check_same_thread=False, isolation_level=None)
conn.execute('PRAGMA journal_mode=WAL;')
conn.executescript('''
CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER);
CREATE TABLE idem (
    key         TEXT PRIMARY KEY,
    status      TEXT NOT NULL,            -- 'IN_PROGRESS' | 'DONE'
    result_json TEXT
);
INSERT INTO accounts VALUES ('alice', 100);
''')

db_lock = threading.Lock()  # SQLite is single-writer; serialize writes in-process

def safe_charge(key, amount):
    # 1. try to claim the key with a single atomic INSERT
    with db_lock:
        try:
            conn.execute(
                "INSERT INTO idem(key, status) VALUES (?, 'IN_PROGRESS')", (key,)
            )
            claimed = True
        except sqlite3.IntegrityError:
            claimed = False

    if not claimed:
        # 2. someone else won — poll until they finish
        for _ in range(200):
            row = conn.execute(
                'SELECT status, result_json FROM idem WHERE key=?', (key,)
            ).fetchone()
            if row and row[0] == 'DONE':
                return ('REPLAY', json.loads(row[1]))
            time.sleep(0.01)
        raise TimeoutError('original request still in progress')

    # 3. we won the claim — do the work, then mark DONE
    try:
        with db_lock:
            conn.execute('BEGIN')
            conn.execute('UPDATE accounts SET balance = balance - ? WHERE name=?', (amount, 'alice'))
            bal = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
            result = {'balance': bal}
            conn.execute(
                "UPDATE idem SET status='DONE', result_json=? WHERE key=?",
                (json.dumps(result), key),
            )
            conn.execute('COMMIT')
        return ('FRESH', result)
    except Exception:
        with db_lock:
            conn.execute('ROLLBACK')
            # release the claim so a later retry can try again
            conn.execute('DELETE FROM idem WHERE key=? AND status=?', (key, 'IN_PROGRESS'))
        raise

k = str(uuid.uuid4())
results = []
def worker():
    results.append(safe_charge(k, 10))

threads = [threading.Thread(target=worker) for _ in range(5)]
for t in threads: t.start()
for t in threads: t.join()

print('results    :', results)
print('FRESH count:', sum(1 for r in results if r[0] == 'FRESH'))
print('REPLAY cnt :', sum(1 for r in results if r[0] == 'REPLAY'))
bal = conn.execute("SELECT balance FROM accounts WHERE name='alice'").fetchone()[0]
print('final bal  :', bal, '← exactly ONE charge applied ✅')


results    : [('FRESH', {'balance': 90}), ('REPLAY', {'balance': 90}), ('REPLAY', {'balance': 90}), ('REPLAY', {'balance': 90}), ('REPLAY', {'balance': 90})]
FRESH count: 1
REPLAY cnt : 4
final bal  : 90 ← exactly ONE charge applied ✅


## 🌍 Real-world flavours of this pattern

| System | Where idempotency lives |
|--------|--------------------------|
| **Stripe API** | `Idempotency-Key` header; 24h retention; `409` if a request with the same key is still in flight. |
| **AWS SQS FIFO** | `MessageDeduplicationId` — duplicates within a 5-minute window are dropped by the broker. |
| **Kafka** | Producer `enable.idempotence=true` attaches a sequence number per partition so the broker drops retried writes. |
| **PayPal / Adyen** | `PayPal-Request-Id` / `idempotencyKey` — same pattern. |
| **GitHub REST** | Many mutations accept an `Idempotency-Key` to make create-or-return-existing safe. |

The machinery differs; the idea is the same: **a stable key + an atomic dedup store + a retention window**.

## 🧠 Takeaways

- *Check-then-act* is broken under concurrency. Make the claim atomic.
- An `IN_PROGRESS` marker lets concurrent retries wait for — or fail fast on — the original request.
- Always free the claim on failure, or a crashed worker will wedge that key forever.